In [19]:
import pandas as pd
import numpy as np

In [20]:
df=pd.read_csv('insurance.csv')

In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


In [22]:
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [23]:
df.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [24]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [25]:
df.groupby('smoker')['charges'].agg(['mean', 'median', 'count'])

,mean,median,count
smoker,,,
no,8434.268298,7345.40530,1064
yes,32050.231832,34456.34845,274


In [26]:
df['bmi_category'] = pd.cut(df['bmi'], bins=[0, 25, 30, 100], labels=['Normal', 'Overweight', 'Obese'])

In [27]:
pivot_bmi_smoke = df.pivot_table(index='bmi_category', columns='smoker', values='charges', aggfunc='mean')
print(pivot_bmi_smoke)

smoker                 no           yes
bmi_category                           
Normal        7547.184007  19839.278309
Overweight    8226.088675  22491.182935
Obese         8853.277294  41692.808992


In [28]:
df['age_group'] = pd.cut(df['age'], bins=[0, 30, 45, 60, 100], labels=['18-30', '31-45', '46-60', '60+'])
print(df.groupby('age_group')['charges'].agg(['mean', 'count']))

                   mean  count
age_group                     
18-30       9397.552051    444
31-45      12647.455654    394
46-60      16340.993846    409
60+        21063.163398     91


In [29]:
print(df.groupby('region')['charges'].mean().sort_values(ascending=False))

region
southeast    14735.411438
northeast    13406.384516
northwest    12417.575374
southwest    12346.937377
Name: charges, dtype: float64


In [30]:
tier_conditions = [
    (df['region'] == 'southeast'),
    (df['region'] == 'northeast')
]

In [31]:
tier_choices = ['Tier 1', 'Tier 2']

In [32]:
df['city_tier'] = np.select(tier_conditions, tier_choices, default='Tier 3')

In [33]:
df.head()

,age,sex,bmi,children,smoker,region,charges,bmi_category,age_group,city_tier
0,19,female,27.900,0,yes,southwest,16884.92400,Overweight,18-30,Tier 3
1,18,male,33.770,1,no,southeast,1725.55230,Obese,18-30,Tier 1
2,28,male,33.000,3,no,southeast,4449.46200,Obese,18-30,Tier 1
3,33,male,22.705,0,no,northwest,21984.47061,Normal,31-45,Tier 3
4,32,male,28.880,0,no,northwest,3866.85520,Overweight,31-45,Tier 3


In [36]:
df['diabetes'] = np.where(
    ((df['smoker'] == 'no') & (df['charges'] > 12000) & (df['bmi'] > 30)) |
    ((df['smoker'] == 'yes') & (df['charges'] > 35000)),
    'Yes', 'No'
)

In [37]:
df['blood_pressure'] = np.where(
    ((df['charges'] > 15000) & (df['age'] > 45)) | 
    ((df['smoker'] == 'yes') & (df['charges'] > 25000)),
    'High', 'Normal'
)

In [41]:
df['adults'] = np.where(df['charges'] > 10000, 2, 1)

In [42]:
df['total_family_members'] = df['adults'] + df['children']

In [43]:
df.head()

,age,sex,bmi,children,smoker,region,charges,bmi_category,age_group,city_tier,diabetes,blood_pressure,adults,total_family_members
0,19,female,27.900,0,yes,southwest,16884.92400,Overweight,18-30,Tier 3,No,Normal,2,2
1,18,male,33.770,1,no,southeast,1725.55230,Obese,18-30,Tier 1,No,Normal,1,2
2,28,male,33.000,3,no,southeast,4449.46200,Obese,18-30,Tier 1,No,Normal,1,4
3,33,male,22.705,0,no,northwest,21984.47061,Normal,31-45,Tier 3,No,Normal,2,2
4,32,male,28.880,0,no,northwest,3866.85520,Overweight,31-45,Tier 3,No,Normal,1,1


In [44]:
def map_family_ages(row):
    ages = [int(row['age'])] # Primary applicant age
    if row['adults'] == 2:
        # Spouse age around primary applicant's age
        ages.append(int(max(18, row['age'] + (row['charges'] % 5 - 2))))
    for i in range(int(row['children'])):
        # Realistic kids ages strictly below 18
        ages.append(int(max(1, min(17, row['age'] - 22 - (i * 2)))))
    return ages

df['family_ages_array'] = df.apply(map_family_ages, axis=1)

In [45]:
df['max_family_age'] = df['family_ages_array'].apply(max)

In [47]:
plan_conditions = [
    (df['max_family_age'] >= 60),                                               
    (df['smoker'] == 'yes') & ((df['diabetes'] == 'Yes') | (df['blood_pressure'] == 'High')), 
    (df['total_family_members'] >= 3) & (df['charges'] > 15000),                
    (df['total_family_members'] == 2) & (df['charges'] > 10000),                
    (df['charges'] > 25000) & (df['bmi'] > 35),                                  
    (df['charges'] >= 8000) & (df['charges'] <= 15000) & (df['city_tier'] == 'Tier 1'), 
]

plan_choices = [
    'Senior Citizen Plan',
    'Critical Illness',
    'Health Guard (Family Floater)',
    'Health Guard (Individual)',
    'Health Care Supreme',
    'Extra Care'
]

In [48]:
df['bajaj_plan'] = np.select(plan_conditions, plan_choices, default='Tax Gain')

In [49]:
print(df['bajaj_plan'].value_counts())

bajaj_plan
Tax Gain                         759
Critical Illness                 162
Health Guard (Individual)        131
Senior Citizen Plan              128
Health Guard (Family Floater)     95
Extra Care                        63
Name: count, dtype: int64


In [50]:
df.head()

,age,sex,bmi,children,smoker,region,charges,bmi_category,age_group,city_tier,diabetes,blood_pressure,adults,total_family_members,family_ages_array,max_family_age,bajaj_plan
0,19,female,27.900,0,yes,southwest,16884.92400,Overweight,18-30,Tier 3,No,Normal,2,2,"[19, 21]",21,Health Guard (Individual)
1,18,male,33.770,1,no,southeast,1725.55230,Obese,18-30,Tier 1,No,Normal,1,2,"[18, 1]",18,Tax Gain
2,28,male,33.000,3,no,southeast,4449.46200,Obese,18-30,Tier 1,No,Normal,1,4,"[28, 6, 4, 2]",28,Tax Gain
3,33,male,22.705,0,no,northwest,21984.47061,Normal,31-45,Tier 3,No,Normal,2,2,"[33, 35]",35,Health Guard (Individual)
4,32,male,28.880,0,no,northwest,3866.85520,Overweight,31-45,Tier 3,No,Normal,1,1,[32],32,Tax Gain


In [51]:
from sklearn.preprocessing import LabelEncoder

In [52]:
from sklearn.model_selection import train_test_split

In [53]:
columns_to_drop = ['charges', 'family_ages_array', 'region', 'age_group', 'bmi_category', 'children', 'adults']

In [54]:
target_col = 'bajaj_plan'

In [55]:
df_clean = df.drop(columns=columns_to_drop)

In [56]:
df_clean.head()

,age,sex,bmi,smoker,city_tier,diabetes,blood_pressure,total_family_members,max_family_age,bajaj_plan
0,19,female,27.900,yes,Tier 3,No,Normal,2,21,Health Guard (Individual)
1,18,male,33.770,no,Tier 1,No,Normal,2,18,Tax Gain
2,28,male,33.000,no,Tier 1,No,Normal,4,28,Tax Gain
3,33,male,22.705,no,Tier 3,No,Normal,2,35,Health Guard (Individual)
4,32,male,28.880,no,Tier 3,No,Normal,1,32,Tax Gain


In [57]:
X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

In [58]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [61]:
categorical_features = ['sex', 'smoker', 'city_tier', 'diabetes', 'blood_pressure']
X_encoded = pd.get_dummies(X, columns=categorical_features, drop_first=True)

In [62]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

In [63]:
from xgboost import XGBClassifier

In [64]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [65]:
xgb_model = XGBClassifier(
    random_state=42, 
    eval_metric='mlogloss',
    use_label_encoder=False 
)

In [66]:
xgb_model.fit(X_train, y_train)

c:\Users\sanskar agrawal\Desktop\pre_rec\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:42:13] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [67]:
y_pred = xgb_model.predict(X_test)

In [68]:
accuracy = accuracy_score(y_test, y_pred)

In [69]:
print(f"\n✅ OVERALL MODEL ACCURACY: {accuracy * 100:.2f}%\n")


✅ OVERALL MODEL ACCURACY: 95.90%



In [70]:
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

                               precision    recall  f1-score   support

             Critical Illness       1.00      1.00      1.00        32
                   Extra Care       0.85      0.85      0.85        13
Health Guard (Family Floater)       0.85      0.89      0.87        19
    Health Guard (Individual)       1.00      0.88      0.94        26
          Senior Citizen Plan       1.00      1.00      1.00        26
                     Tax Gain       0.96      0.97      0.97       152

                     accuracy                           0.96       268
                    macro avg       0.94      0.93      0.94       268
                 weighted avg       0.96      0.96      0.96       268



In [73]:
import joblib

In [74]:
joblib.dump(xgb_model, 'bajaj_xgboost_model.pkl')

['bajaj_xgboost_model.pkl']

In [75]:
joblib.dump(label_encoder, 'bajaj_label_encoder.pkl')

['bajaj_label_encoder.pkl']

In [76]:
model_columns = X_train.columns.tolist()
joblib.dump(model_columns, 'model_columns.pkl')

['model_columns.pkl']